In [27]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import build_preprocessor

PATH_NAME = Path("../data/processed/spaceship_titanic_feature_engineered.csv")
df = pd.read_csv(PATH_NAME)
df.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,...,Group number,Passengers within group,Total spend,Group size,Is alone,No spending,Luxury spending,Basic spending,Spending per person,Transported
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,...,1,1,0.0,1,True,True,0.0,0.0,0.0,False
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,...,2,1,736.0,1,True,False,593.0,34.0,736.0,True
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,...,3,1,10383.0,2,False,False,6764.0,3576.0,5191.5,False
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,...,3,2,5176.0,2,False,False,3522.0,1654.0,2588.0,False
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,...,4,1,1091.0,1,True,False,567.0,221.0,1091.0,True


In [28]:
# splitting into training and test sets
from sklearn.model_selection import train_test_split

X = df.drop("Transported", axis=1)
y = df.Transported

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [29]:
# select numerical and categorical features
numFeatures = X.select_dtypes(include="number").columns
catFeatures = [col for col in X.columns if col not in numFeatures]

In [30]:
# creating pipelines
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

preprocessor = build_preprocessor(numFeatures, catFeatures)

pipelines = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    "SVM": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", SVC(kernel="rbf", probability=True))
    ]),     
}

In [31]:
# train models and calculate metrics
import joblib
from sklearn.metrics import (accuracy_score, precision_score, recall_score, roc_auc_score)

results = {}

for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    if hasattr(pipeline, "predict_proba"):
        y_probs = pipeline.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_probs)
    else:
        roc_auc = None
    
    results[model_name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "ROC AUC": roc_auc
    }
    
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [32]:
# save metrics
results_df = pd.DataFrame(results).T.round(3)
results_df.to_csv("../data/processed/model_metrics.csv")